<a href="https://colab.research.google.com/github/aishwaryasuriyakumar/aishwaryasrcs09-codeboosters-internship-2026/blob/main/Day_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence_transformers chromadb groq pandas -q

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

import os
print("All libraries imported successfully !")
print("Ready to buld a RAG system !")

All libraries imported successfully !
Ready to buld a RAG system !


In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

!pip install groq
from groq import Groq
import os
print("All libraries imported successfully !")
print("Ready to buld a RAG system !")

All libraries imported successfully !
Ready to buld a RAG system !


In [ ]:
GROQ_API_KEY = "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"

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized.")
print("NOTE : If you see an authentication error later, double-check your API KEY.")

Groq API client initialized.
NOTE : If you see an authentication error later, double-check your API KEY.


In [ ]:
df = pd.read_csv('college_notes.csv')

print("Shape of dataset : ", df.shape)
print("\nColumn names : ", df.columns.tolist())

Shape of dataset :  (15, 4)

Column names :  ['note_id', 'subject', 'topic', 'content']


In [ ]:
#EXPLORE THE DATASET

print("Subjects in the dataset : ")
print(df['subject'].value_counts())
print("\nSample of topics : ")
print(df[['note_id', 'subject', 'topic']].to_string(index=False))
print("\nLength of content (number of characters) for each note: ")

df['content_length'] = df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index=False))

Subjects in the dataset : 
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics : 
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Py

In [ ]:
print("Loading embedding model...")
print("this is may take 30 to 60 secs")
print("subsequence run will be faster as model cached")
embedding_model=SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully")

test_embedding = embedding_model.encode("This is a test sentence.")
print(f"Test embedding shape : {test_embedding.shape}")
print(f"First 5 values of test embedding: {test_embedding[:5]}')")

Loading embedding model...
this is may take 30 to 60 secs
subsequence run will be faster as model cached


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully
Test embedding shape : (384,)
First 5 values of test embedding: [0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]')


In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name = "college_notes_rag")

print("ChromaDB client created.")
print(f"Collection name : college_notes_rag")
print(f"Documents in collection so far : {collection.count()}")

ChromaDB client created.
Collection name : college_notes_rag
Documents in collection so far : 15


In [ ]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict(orient='records')]
metadatas=[
    {"subject":row['subject'],"topic":row['topic']} for row in df.to_dict(orient='records')
]
print(f"Total chunk prepared:{len(documents)}")
print(f"First Documnet ID :{ids[0]}")
print(f"First metadata :{metadatas[0]}")

Total chunk prepared:15
First Documnet ID :note_N001
First metadata :{'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}


In [ ]:
print("Generating embeddings for all 15 notes...")
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\nEmbedding matrix shape : {embeddings.shape}")
embeddings_list = embeddings.tolist()

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list
)

print(f"\nDocumnets successfully added to ChromaDB.")
print(f"Total documents in collection : {collection.count()}")

Generating embeddings for all 15 notes...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape : (15, 384)

Documnets successfully added to ChromaDB.
Total documents in collection : 15


In [ ]:
#create retreival function

def retrieve_relevant_chunks(question, top_k = 3):
  """
  Given a user question, retrieves the most relevant document chunks from ChromaDB.

  Paramaters:
    question (str) : The user's question as a text string
    top_k    (int) : How many top results to return(default : 3)

  Returns :
    A dictionary containing retrived documents, distances, and metadata
  """
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(query_embeddings=[question_embedding], n_results=top_k)
  return results

In [ ]:
test_question="what is ETL and how does it work in data engineering"
print(f"test question:{test_question}")
print("="*60)

results =retrieve_relevant_chunks(test_question, top_k=3)

print("\top 3 retrived chunks")
print("="*60)

for i,(doc,dist,meta) in enumerate(zip(results['documents'][0],results['distances'][0],results['metadatas'][0])):
  print(f"results:{i+1}")
  print(f"subject:{meta['subject']}")
  print(f"topic:{meta['topic']}")
  print(f"distance:{dist:.4f}")
  print(f"content:{doc[:120]}...")

test question:what is ETL and how does it work in data engineering
	op 3 retrived chunks
results:1
subject:Data Engineering
topic:ETL Pipelines
distance:0.2041
content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...
results:2
subject:Data Engineering
topic:APIs and Data Collection
distance:1.1100
content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...
results:3
subject:Python Programming
topic:Data Visualization
distance:1.3892
content:Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


THE RAG PROMPT TEMPLATE

In [ ]:
def build_context_from_results(results):
    """
    Format ChromaDB retrieval results into a context string.
    """

    context_parts = []

    for i, (doc, meta) in enumerate(
        zip(results['documents'][0], results['metadatas'][0])
    ):
        context_parts.append(f"Subject: {meta['subject']}")
        context_parts.append(f"Topic: {meta['topic']}")

        chunk_text = (
            f"[Source {i+1}: {meta['subject']} - {meta['topic']}]\n{doc}"
        )
        context_parts.append(chunk_text)

    context_str = "\n\n---\n\n".join(context_parts)

    return context_str

context = build_context_from_results(results)
print("Built context string from retreived chunks :")
print("="*60)
print(context[:500] + "...")
print(f"\nTotal context length : {len(context)} characters")

Built context string from retreived chunks :
Subject: Data Engineering

---

Topic: ETL Pipelines

---

[Source 1: Data Engineering - ETL Pipelines]
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

---

Subject: Data Engineering

---

Topic: APIs and Data Collection

---

[Source 2: Data Engineering - APIs and Data Collection]
An API or Application Programming Interface all...

Total context length : 1051 characters


In [ ]:
def generate_rag_answer(question, context):
    """
    Generate an answer to a question using a RAG approach.

    parameters :
      question (str) : The question to be answered.
      context (str) : The context from which to retrieve relevant information.

    """
    system_prompt = """You are a helpful academic assistant for engineering students
    you will be given context of retreived from a college knowledge base.

    RULES:
    1. Answer only using the information provided in the context table
    2.If the answer is not found in the context, say exactly:
    "I dont have enough information to answer this question"
    3.Do not use your own knowledge to answer
    4.Keep answers clear, accure, and beginner-friendly
    5.Mention which source the information came from whe ppossible. """

    user_prompt  = f"""Context from knowledge base :

{context}

---
Students Question : {question}
please answer the folllwing question using the context above """

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature = 0.1,
        max_tokens = 500
    )
    answer = response.choices[0].message.content
    return answer


print("RAG generation function defined.")

RAG generation function defined.


In [ ]:
def ask_college_assistant(question,top_k=3,verbose=True):
  """
  Complete RAG pipeline:Given a question ,retrieve relevant context and generate an
  Parameters:
  question (str) :The user's question
  top_k  (int) :Number of chunks to retrieve (Default:3)
  verbose (bool) :Whether to print intermediate strps(default:True)
  Returns:
  answer (str) :The final generated answer
  """
  if verbose:
    print(f"Question:{question}")
    print("="*60)
    print("Step 1:Retrieving relevant documents:")
  results=retrieve_relevant_chunks(question,top_k)
  if verbose:
    print(f"Retrieved {top_k} chunks from the knowledge base:")
    for i ,meta in enumerate (results['metadatas'][0]):
      print(f"{i+1}.{meta['subject']}-{meta['topic']}")
    print("\n Step 2:Building context string...")
  context=build_context_from_results(results)
  if verbose:
    print(f"Context built ({len(context)})characters)")
    print("\n Step 3:Sending to LLM for answer generation...")
  answer=generate_rag_answer(question,context)
  if verbose:
    print("\n"+ "=" *60)
    print("ANSWER")
    print("=" *60)

  return answer
print("Complete RAG pipeline function ready.")
print("Function :ask_college_assistent(question,top_k=3)")

Complete RAG pipeline function ready.
Function :ask_college_assistent(question,top_k=3)


In [ ]:
question1 = "What is ETL and what are its three main stages?"
answer1 = ask_college_assistant(question1, top_k=3, verbose=True)
print(answer1)

Question:What is ETL and what are its three main stages?
Step 1:Retrieving relevant documents:
Retrieved 3 chunks from the knowledge base:
1.Data Engineering-ETL Pipelines
2.Generative AI-Retrieval Augmented Generation
3.Generative AI-Prompt Engineering

 Step 2:Building context string...
Context built (1123)characters)

 Step 3:Sending to LLM for answer generation...

ANSWER
According to [Source 1: Data Engineering - ETL Pipelines], ETL stands for Extract Transform Load. 

The three main stages of ETL are:

1. **Extract**: Collecting raw data from different sources.
2. **Transform**: Transforming the raw data into a clean and structured format.
3. **Load**: Loading the transformed data into a database or data warehouse for analysis.

These stages are the core components of the ETL process.


In [ ]:
question2="How do embeddings help in building search systems?"
answer_2=ask_college_assistant(question2,top_k=3,verbose=True)

Question:How do embeddings help in building search systems?
Step 1:Retrieving relevant documents:
Retrieved 3 chunks from the knowledge base:
1.Generative AI-Retrieval Augmented Generation
2.Generative AI-Large Language Models
3.Machine Learning-Feature Engineering

 Step 2:Building context string...
Context built (1112)characters)

 Step 3:Sending to LLM for answer generation...

ANSWER


In [ ]:
question3="What is the population of  Tokyo?"
print("Testing with an out-of scope question (not in college notes):")
answer_3=ask_college_assistant(question3,top_k=3,verbose=True)


Testing with an out-of scope question (not in college notes):
Question:What is the population of  Tokyo?
Step 1:Retrieving relevant documents:
Retrieved 3 chunks from the knowledge base:
1.Generative AI-Large Language Models
2.Data Engineering-SQL Databases
3.Data Engineering-Data Cleaning

 Step 2:Building context string...
Context built (981)characters)

 Step 3:Sending to LLM for answer generation...

ANSWER


MINI - PROJECT
   Project Description
   Build a complete Knowledge Assistant that:
      1. Loads the college_notes.csv knowledge base
      2. Indexes all notes in ChromaDB with embeddings
      3. Accepts a student question
      4. Retrives the top 3 relvant roles
      5. Injects

In [ ]:
# MINI PROJECT - College Knowledge Assistant Demo

print("""
==========================================
COLLEGE KNOWLEDGE ASSISTANT DEMONSTRATION
==========================================
""")

# --- Test Case 1: In-scope question ---
print("\n--- Test Case 1: In-scope question with expected grounded answer ---")

question_demo_1 = "What are the main types of supervised learning models?"
answer_demo_1 = ask_college_assistant(question_demo_1, top_k=3, verbose=True)

print(answer_demo_1)

# --- Test Case 2: Out-of-scope question ---
print("\n--- Test Case 2: Out-of-scope question ---")

question_demo_2 = "Who was the first person to walk on the moon?"
answer_demo_2 = ask_college_assistant(question_demo_2, top_k=3, verbose=True)

print(answer_demo_2)

# --- Test Case 3: Generative AI Question ---
print("\n--- Test Case 3: Generative AI Question ---")

question_demo_3 = "Explain what Large Language Models are."
answer_demo_3 = ask_college_assistant(question_demo_3, top_k=3, verbose=True)

print(answer_demo_3)

print("\n==========================================")
print("ASSISTANT DEMONSTRATION COMPLETE")
print("==========================================")


COLLEGE KNOWLEDGE ASSISTANT DEMONSTRATION


--- Test Case 1: In-scope question with expected grounded answer ---
Question:What are the main types of supervised learning models?
Step 1:Retrieving relevant documents:
Retrieved 3 chunks from the knowledge base:
1.Machine Learning-Supervised Learning
2.Generative AI-Large Language Models
3.Machine Learning-Decision Trees

 Step 2:Building context string...
Context built (1058)characters)

 Step 3:Sending to LLM for answer generation...

ANSWER
According to the context, the main types of supervised learning models are not explicitly mentioned. However, it is mentioned that supervised learning includes examples such as classification and regression. 

Therefore, the answer is:
Classification and Regression. 

[Source: Machine Learning - Supervised Learning]

--- Test Case 2: Out-of-scope question ---
Question:Who was the first person to walk on the moon?
Step 1:Retrieving relevant documents:
Retrieved 3 chunks from the knowledge base:
1.Dat